# PV2H 演示：数据 → 训练 → 预测 → 曲线 → 南网回测

本 Notebook 用于**展示运行流程**（赛题为 **3 个场站** S01–S03；下表汇总行为第 4 行「三站等权」）。

南网两项评价指标（`TASK.md` §6.3，`tools/evaluate.py` 输出）：
1. **场站南网日均准确率** \(A_s\)
2. **三站等权南网日均准确率** \(\mathrm{Score}\)

运行前：在仓库根目录安装依赖，并将 `DATA_ROOT` 指向含 `cloud/`、`power/`、`tools/` 的数据包目录。

In [ ]:
from pathlib import Path
import os
import sys
import json

import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pv2h_model import config
from pv2h_model.data import load_stations, make_release
from pv2h_model.eval.reporting import (
    curve_frame,
    nanwang_report_table,
    official_score,
    predictions_from_samples,
    score_rows_metric,
)
from predict import predict, get_predictor

# ---------- 可改参数 ----------
DATA_ROOT = Path(os.environ.get("PV2H_DATA", config.DATA_ROOT))
SPLIT = "val"  # train | val

# 训练：首次演示建议 False（使用仓库自带 models/）；需要重训时改 True
RUN_TRAIN = False
TRAIN_STAGES = "stats,samples"  # 不含 members（耗时数小时）；全量: "all"

# 在线起报演示（慢，读云图 ZIP）
DEMO_ISSUE = "09-01 02:00"

# 绘图：目标功率时刻区间（UTC MM-DD HH:mm）
PLOT_STATION = "S01"
PLOT_LEAD = 60
PLOT_TARGET_START = "09-01 08:00"
PLOT_TARGET_END = "09-01 18:00"

print("ROOT:", ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("models:", config.MODELS_DIR.exists(), "blend:", (config.MODELS_DIR / "blend.json").exists())

## 1. 数据读取

In [ ]:
stations = load_stations()
power = pd.read_csv(
    DATA_ROOT / "power" / f"{SPLIT}.csv.gz",
    dtype={"station_id": str},
    float_precision="round_trip",
)
print("场站:", stations)
print("功率记录:", power.shape, "时间范围:", power.timestamp.min(), "~", power.timestamp.max())

with make_release(DATA_ROOT) as ds:
    demo_in = ds.inputs(
        DEMO_ISSUE,
        power_steps=config.POWER_STEPS,
        cloud_frames=config.CLOUD_FRAMES,
        bands=config.CLOUD_BANDS,
    )
print("起报:", DEMO_ISSUE, "功率窗口:", demo_in["power_ratio"].shape, "云帧:", next(iter(demo_in["cloud"].values())).shape)

## 2. 训练（可选）

默认使用 Git 仓库中的 `models/`。若 `RUN_TRAIN=True`，调用 `train.py` 按阶段重建统计量 / 样本 / 成员。

In [ ]:
if RUN_TRAIN:
    for stage in TRAIN_STAGES.split(","):
        stage = stage.strip()
        print("=== train stage:", stage, "===")
        subprocess.run(
            [sys.executable, str(ROOT / "train.py"), "--stage", stage],
            cwd=str(ROOT),
            check=True,
        )
else:
    print("跳过训练，加载已有模型:", json.loads((config.MODELS_DIR / "blend.json").read_text())["members"])

bp = get_predictor()
print("融合成员:", bp.members)

## 3. 预测

- **在线**：单次起报 `predict(inputs)`（与提交包一致）
- **离线快路径**：在存在 `cache/samples_{split}.parquet` 时，对整表特征一次融合（用于回测与绘图）

In [ ]:
online_pred = predict(demo_in)
display(online_pred.head(8))

samples_path = config.CACHE_DIR / f"samples_{SPLIT}.parquet"
if not samples_path.exists():
    raise FileNotFoundError(
        f"缺少 {samples_path}，请先 RUN_TRAIN 且 TRAIN_STAGES 含 samples，或运行: python train.py --stage samples"
    )

samples = pd.read_parquet(samples_path)
offline_pred = bp.predict_frame(samples)
print("离线样本预测:", len(offline_pred), "行")

## 4. 预测 vs 实测曲线（可设定目标时刻区间）

In [ ]:
curve = curve_frame(
    samples,
    offline_pred,
    station_id=PLOT_STATION,
    lead_minutes=PLOT_LEAD,
    target_start=PLOT_TARGET_START,
    target_end=PLOT_TARGET_END,
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(curve["target_time"], curve["y_true"], label="实测 power_ratio", linewidth=1.2)
ax.plot(curve["target_time"], curve["pred"], label=f"预测 lead={PLOT_LEAD}min", linewidth=1.2, alpha=0.85)
if "clearsky_target" in curve.columns:
    ax.plot(curve["target_time"], curve["clearsky_target"], label="clearsky_target", linestyle="--", alpha=0.5)
ax.set_title(f"{PLOT_STATION} 目标时刻 {PLOT_TARGET_START} ~ {PLOT_TARGET_END} (UTC)")
ax.set_xlabel("target_time (UTC)")
ax.set_ylabel("capacity-normalized power")
ax.legend(loc="upper right")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
print("有效点数:", len(curve))

## 5. 回测：南网精度（官方 + 行级对照）

生成完整验证集预测表并调用 `tools/evaluate.py` 同口径打分。

In [ ]:
pred_df = predictions_from_samples(samples, offline_pred)
official = official_score(pred_df, DATA_ROOT, split=SPLIT)
rows_metric = score_rows_metric(samples, offline_pred)

report = nanwang_report_table(official)
display(report)

print("\n官方 JSON（与 evaluate.py 一致）:")
print(official)
print("\n行级 score_rows 对照（应接近官方分）:", rows_metric)

### 三分场站明细 + 等权汇总（两列南网指标）

In [ ]:
per = official["station_daily_accuracy_percent"]
total = official["score_percent"]
detail = pd.DataFrame(
    {
        "统计对象": list(config.STATION_IDS) + ["三站等权"],
        "场站南网日均准确率_%": [per[s] for s in config.STATION_IDS] + [float("nan")],
        "三站等权南网日均准确率_%": [float("nan")] * len(config.STATION_IDS) + [total],
    }
)
detail["场站南网日均准确率_%"] = detail["场站南网日均准确率_%"].round(4)
detail["三站等权南网日均准确率_%"] = detail["三站等权南网日均准确率_%"].round(4)
display(detail.style.hide(axis="index"))